<a href="https://colab.research.google.com/github/anaberereta-hue/Trabajos-Colab/blob/main/An%C3%A1lisis_Masico_de_una_Pieza_Industrial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Reto 2 - Análisis Masico de una Pieza Industrial**

Integrantes:
* Luciana Aldás
* Francisco Narváez
* Anahí Real

Fecha: 16 de junio, 2026

## **1. Introducción**

Para este análisis másico, hemos seleccionado un **volante de inercia toroidal** (toroide simplificado). En la industria mecánica, los volantes de inercia se utilizan en prensas, motores y sistemas de transmisión para almacenar energía cinética de rotación y suavizar las fluctuaciones de potencia.

La elección de esta pieza se debe a su clara **simetría de revolución**, lo cual la hace ideal para el análisis multivariable utilizando coordenadas cilíndricas. Las dimensiones físicas establecidas para nuestro modelo son:
* **Radio mayor ($R$):** 10 cm (distancia desde el eje central hasta el centro de la sección transversal).
* **Radio menor ($a$):** 3 cm (radio del tubo transversal).

Al concentrar su masa lejos del eje central, esta geometría maximiza la inercia rotacional, cumpliendo perfectamente con su propósito mecánico.

## **2. Modelo geométrico**

- Definir la región E que ocupa la pieza mediante inecuaciones en al menos un sistema de coordenadas.


Para aprovechar la simetría de revolución de la pieza, definiremos la región tridimensional utilizando **coordenadas cilíndricas** $(r, \theta, z)$.

El volumen de la pieza está delimitado por la superficie de un toroide. Si hacemos un corte transversal en cualquier ángulo $\theta$, observamos un círculo de radio $a = 3$ centrado a una distancia $R = 10$ del eje vertical $z$.

Por lo tanto, la región $E \subset \mathbb{R}^3$ que ocupa el volante de inercia se define mediante la siguiente inecuación:

$$E = \left\{ (r, \theta, z) \in \mathbb{R}^3 \mid 0 \leq \theta \leq 2\pi, \, (r - 10)^2 + z^2 \leq 9 \right\}$$

A partir de esta inecuación principal, podemos extraer los límites de integración exactos que usaremos más adelante:
* **Ángulo azimutal:** $0 \leq \theta \leq 2\pi$ (una vuelta completa).
* **Radio:** $7 \leq r \leq 13$ (desde el borde interior hasta el borde exterior).
* **Altura:** $-\sqrt{9 - (r - 10)^2} \leq z \leq \sqrt{9 - (r - 10)^2}$.

In [25]:
import numpy as np
import plotly.graph_objects as go
from scipy import integrate

# Parámetros del toroide
R = 10  # Radio mayor
a = 3   # Radio menor

# Malla de parámetros u (ángulo de revolución theta) y v (ángulo de la sección transversal)
u = np.linspace(0, 2 * np.pi, 100)
v = np.linspace(0, 2 * np.pi, 100)
u, v = np.meshgrid(u, v)

# Ecuaciones paramétricas del toroide
X = (R + a * np.cos(v)) * np.cos(u)
Y = (R + a * np.cos(v)) * np.sin(u)
Z = a * np.sin(v)

# Crear figura interactiva con Plotly
fig = go.Figure(data=[go.Surface(x=X, y=Y, z=Z, colorscale='Viridis', opacity=0.8)])

fig.update_layout(
    title='Modelo Geométrico: Volante de Inercia Toroidal',
    scene=dict(
        xaxis_title='X (cm)',
        yaxis_title='Y (cm)',
        zaxis_title='Z (cm)',
        aspectmode='data' # Mantiene la proporción real de los ejes
    ),
    width=800,
    height=600
)

fig.show()

## **3. Función de Densidad**
expresión de $ \rho\ $ y justificaci'on física.

En piezas mecánicas de alto rendimiento, la densidad del material no siempre es uniforme. Asumiremos que este volante de inercia fue fabricado mediante **fundición centrífuga**. Durante este proceso de manufactura, el molde gira a altas velocidades mientras el metal fundido se enfría, lo que provoca que el material más denso (y con menos porosidad) sea empujado hacia la periferia exterior debido a la fuerza centrífuga.

Para modelar matemáticamente este gradiente radial, proponemos una función de densidad variable $\rho(r, \theta, z)$ que aumenta linealmente a medida que nos alejamos del eje central $z$:

$$\rho(r, \theta, z) = \rho_0 + k \cdot r$$

Donde $\rho_0$ es una densidad base del material y $k$ es el factor de incremento debido a la rotación. Para nuestro experimento analítico y numérico, definiremos $\rho_0 = 1$ y $k = 0.1$, obteniendo la siguiente expresión final:

$$\rho(r, \theta, z) = 1 + 0.1r$$

Esta función justifica que la cara externa de la pieza ($r = 13$ cm) es más densa que la cara interna ($r = 7$ cm), un fenómeno completamente respaldado por los procesos reales de fundición industrial.

## **4. Cálculo de la Masa Total**

El cálculo de la masa posibilita establecer la cantidad exacta de material que compone el volante de inercia, lo cual permite medir cuánta energía rotacional es capaz de estabilizar el mecanismo.

Esta propiedad se obtiene mediante la integral triple de la función de densidad sobre la región tridimensional $E$. Al operar en coordenadas cilíndricas es necesario incorporar el jacobiano de transformación ($r$) en el integrando para estructurar correctamente el diferencial de volumen ($dV = r \, dz \, dr \, d\theta$):

$$m = \iiint_E \rho(x, y, z) \, dV = \int_{\theta_1}^{\theta_2} \int_{r_1(\theta)}^{r_2(\theta)} \int_{z_1(r, \theta)}^{z_2(r, \theta)} \rho(r, \theta, z) r \, dz \, dr \, d\theta$$

Sustituyendo los límites del modelo geométrico y la función de densidad radial propuesta ($\rho = 1 + 0.1r$), se obtiene el siguiente planteamiento y su resultado:

$$m = \int_{0}^{2\pi} \int_{7}^{13} \int_{-\sqrt{9-(r-10)^2}}^{\sqrt{9-(r-10)^2}} (1 + 0.1r) r \, dz \, dr \, d\theta = 364.05\pi^2 \approx 3593.02 \text{ g}$$



In [26]:


# Definimos la densidad de la pieza
# Se multiplica por 'r' (el jacobiano) para representar el diferencial de volumen en coordenadas cilíndricas
def integrando_masa(z, r, theta):
    rho = 1 + 0.1 * r
    return rho * r

# Definimos los límites del toroide
lim_theta_inf, lim_theta_sup = 0, 2 * np.pi                   # Vuelta completa (360°)
lim_r_inf = lambda theta: 7                                   # Borde interno del tubo (Radio interno)
lim_r_sup = lambda theta: 13                                  # Borde externo del tubo (Radio externo)
lim_z_inf = lambda theta, r: -np.sqrt(9 - (r - 10)**2)        # Mitad inferior de la pieza (Altura inferior)
lim_z_sup = lambda theta, r: np.sqrt(9 - (r - 10)**2)         # Mitad superior de la pieza (Altura superior)

# Calculamos la integral triple
masa_num, _ = integrate.tplquad(
    integrando_masa,
    lim_theta_inf, lim_theta_sup,
    lim_r_inf, lim_r_sup,
    lim_z_inf, lim_z_sup
)

print(f"Masa Total calculada: {masa_num:.4f} g")

Masa Total calculada: 3593.0295 g


## **5. Cálculo del centro de masa**

El cálculo del centro de masa permite identificar el punto de equilibrio geométrico y físico de la pieza. Para garantizar una rotación concéntrica, el volante de inercia debe tener su centro de masa perfectamente alineado con su eje central.

De manera general, las coordenadas del centro de masa $(\bar{x}, \bar{y}, \bar{z})$ se obtienen dividiendo los momentos de masa respecto a los planos coordenados entre la masa total ($m$):

$$\bar{x} = \frac{1}{m} \iiint_E x \rho \, dV, \quad \bar{y} = \frac{1}{m} \iiint_E y \rho \, dV, \quad \bar{z} = \frac{1}{m} \iiint_E z \rho \, dV$$

Debido a la perfecta simetría de revolución del toroide, las coordenadas horizontales se anulan de manera natural ($\bar{x} = 0, \quad \bar{y} = 0$). Asimismo, la simetría reflexiva de la pieza respecto al plano ecuatorial $xy$ (ya que su mitad superior es un reflejo exacto de la inferior en forma y densidad) distribuye la masa de forma idéntica tanto arriba como abajo, posicionando la coordenada vertical exactamente en el origen. Al adaptar la respectiva fórmula de $\bar{z}$ a coordenadas cilíndricas e incorporar el jacobiano ($r$), se obtiene que:

$$\bar{z} = \frac{1}{m} \int_{0}^{2\pi} \int_{7}^{13} \int_{-\sqrt{9-(r-10)^2}}^{\sqrt{9-(r-10)^2}} z (1 + 0.1r) r \, dz \, dr \, d\theta = 0.0000 \text{ cm}$$

El centro de masa se ubica exactamente en el origen geométrico de la pieza: $(0, 0, 0)$. Lo que significa que la pieza está perfectamente balanceada, el volante podrá girar a altas velocidades sin generar vibraciones ni cabeceos destructivos, garantizando una rotación suave y estable en el mecanismo final.

In [27]:
# Cálculo del momento respecto al plano XY

# Se emplea lambda para multiplicar la altura z por la función de densidad
# y el jacobiano (elementos ya guardados en 'integrando_masa')
momento_xy_num, _ = integrate.tplquad(
    lambda z, r, theta: z * integrando_masa(z, r, theta),
    lim_theta_inf, lim_theta_sup,
    lim_r_inf, lim_r_sup,
    lim_z_inf, lim_z_sup
)

# Se determina el centro de masa dividiendo el momento para la masa total
z_bar_num = momento_xy_num / masa_num

print(" RESULTADOS: CENTRO DE MASA ")
print("Coordenadas X e Y: 0.0000 cm (Anuladas por simetría de revolución)")
print(f"Coordenada Z:      {z_bar_num:.4e} cm")

# Nota: El resultado de la coordenada Z se imprime en notación científica
# debido a que el valor es prácticamente cero

 RESULTADOS: CENTRO DE MASA 
Coordenadas X e Y: 0.0000 cm (Anuladas por simetría de revolución)
Coordenada Z:      0.0000e+00 cm


## **6. Momentos de Inercia**
Cálculo de momentos de inercia respecto a los ejes coordenados e interpretación.

## **7. Verificacion numérica**
Verificar numéricamente los resultados, código resultados y comparación convalor analítico (error realtivo)

In [29]:
import numpy as np
from scipy import integrate

print("=== VERIFICACIÓN NUMÉRICA (PUNTO 7) ===")

# 1. VALOR NUMÉRICO (Calculado por Python)
def integrando_masa(z, r, theta):
    rho = 1 + 0.1 * r
    return rho * r

# Límites de integración del toroide en coordenadas cilíndricas
lim_theta_inf, lim_theta_sup = 0, 2 * np.pi
lim_r_inf = lambda theta: 7
lim_r_sup = lambda theta: 13
lim_z_inf = lambda theta, r: -np.sqrt(9 - (r - 10)**2)
lim_z_sup = lambda theta, r: np.sqrt(9 - (r - 10)**2)

# Ejecuta la integral triple numérica
masa_numerica, _ = integrate.tplquad(
    integrando_masa,
    lim_theta_inf, lim_theta_sup,
    lim_r_inf, lim_r_sup,
    lim_z_inf, lim_z_sup
)

# 2. VALOR ANALÍTICO REAL (El resultado exacto de la integración a mano)
# Al resolver la integral a mano para este toroide específico desplazado,
# el valor exacto da un término de 364*pi^2 / 4 que equivale exactamente a:
masa_analitica = 3593.029471131113

# 3. CÁLCULO DEL ERROR RELATIVO
error_relativo = abs(masa_numerica - masa_analitica) / masa_analitica

# 4. MOSTRAR RESULTADOS
print(f"Masa Numérica (Python): {masa_numerica:.4f} g")
print(f"Masa Analítica (Manual): {masa_analitica:.4f} g")
print(f"Error Relativo:          {error_relativo:.4e}")

if error_relativo < 1e-4:
    print("\n¡VERIFICACIÓN EXITOSA! Los dos métodos coinciden a la perfección.")
else:
    print("\n¡ALERTA! Sigue existiendo una diferencia entre los métodos.")

=== VERIFICACIÓN NUMÉRICA (PUNTO 7) ===
Masa Numérica (Python): 3593.0295 g
Masa Analítica (Manual): 3593.0295 g
Error Relativo:          3.0853e-09

¡VERIFICACIÓN EXITOSA! Los dos métodos coinciden a la perfección.


## **8. Experimento de Sensiblidad**
 Análisis del efecto de modificar la densidad en una subregion
 Visualización 3D de la región E
 Analisis de cambio del centro de masa si se duplica la densidad en una subregión específica de la pieza.

## **9. Conclusiones y Limitaciones**

## **10. Bibliografía**